<a id="camera-basis"></a>

$$
\text{相机坐标系 (Camera Basis): 由 lookAt(eye, target, up) 构造三个互相垂直的单位向量} \\
\text{eye: 相机位置, target: 看向的点, up: 大致的 "上" 方向 (例如世界的 y 轴, 不需要和视线垂直)} \\ \quad \\
\text{约定 (OpenGL): 相机看向自己的 -z 方向, x 向右, y 向上} \\
\text{1. 相机 z 轴 (指向相机后方): } w = \frac{eye - target}{|eye - target|} \\
\text{2. 相机 x 轴 (右): 要同时垂直于 up 和 w, 用叉积 } r = \frac{up \times w}{|up \times w|} \\
\text{3. 相机 y 轴 (上): 垂直于 w 和 r, } v = w \times r \text{ (w, r 已经是垂直的单位向量, 叉积长度 = 1, 不用再归一化)} \\ \quad \\
\text{和 Gram-Schmidt 的关系: v 其实就是 up 去掉在 w 方向上的分量再归一化} \\
v = \frac{up - (up \cdot w) \, w}{|up - (up \cdot w) \, w|} \\
\text{(用二重叉积公式验证: } w \times (up \times w) = up \, (w \cdot w) - w \, (w \cdot up) = up - (up \cdot w) w \text{, 方向一致)} \\ \quad \\
\text{右手系检查: } r \times v = w \\
\text{退化情况: up 和视线平行时 } up \times w = 0 \text{, 无法确定 "右" 方向 (相机正对天顶时要换一个 up)}
$$

数学基础: [Gram-Schmidt 与 QR 分解](../math_basics/linear_algebra.ipynb#qr) · [正交矩阵](../math_basics/linear_algebra.ipynb#orthogonal) · [投影](../math_basics/linear_algebra.ipynb#projection)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True, precision=4)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

def normalize(v):
    return v / np.linalg.norm(v)

def camera_basis(eye, target, up):
    w = normalize(eye - target)
    cross = np.cross(up, w)
    if np.linalg.norm(cross) < 1e-9:
        raise ValueError('up 和视线平行, 无法确定相机的右方向')
    r = normalize(cross)
    v = np.cross(w, r)
    return r, v, w

eye = np.array([4.0, 3.0, 5.0])
target = np.array([0.0, 0.5, 0.0])
up = np.array([0.0, 1.0, 0.0])
r, v, w = camera_basis(eye, target, up)
print(f'r = {r}, v = {v}, w = {w}')
R_cam = np.column_stack([r, v, w])
print(f'R^T R == I ? {np.allclose(R_cam.T @ R_cam, np.eye(3))}, det = {np.linalg.det(R_cam):.6f}, r x v == w ? {np.allclose(np.cross(r, v), w)}')

# Gram-Schmidt 版本的 v
v_gs = normalize(up - (up @ w) * w)
print(f'叉积得到的 v == Gram-Schmidt 得到的 v ? {np.allclose(v, v_gs)}')
# QR 分解: [w, up] 的 Q 的两列就是对 (w, up) 做 Gram-Schmidt, 可能差一个符号
Q, Rq = np.linalg.qr(np.column_stack([w, up]))
v_qr = Q[:, 1] * np.sign(Rq[1, 1])
print(f'QR 分解得到的 v == v ? {np.allclose(v_qr, v)}')

try:
    camera_basis(np.array([0, 5.0, 0]), np.zeros(3), up)
except ValueError as e:
    print(f'相机正上方往下看: {e}')

<a id="view-matrix"></a>

$$
\text{视图矩阵 (View Matrix): 把世界坐标变成相机坐标} \\ \quad \\
\text{先写相反方向: 相机坐标 → 世界坐标 (camera-to-world)} \\
\text{相机坐标里的点 } (a, b, c) \text{ 在世界里的位置 } = eye + a \, r + b \, v + c \, w \\
M_{c2w} = \begin{bmatrix} r & v & w & eye \\ 0 & 0 & 0 & 1 \end{bmatrix} = \begin{bmatrix} R & eye \\ 0 & 1 \end{bmatrix} \quad (\text{R 的三列是 r, v, w}) \\ \quad \\
\text{视图矩阵是它的逆, 用仿射逆公式, R 是正交矩阵所以 } R^{-1} = R^T \text{:} \\
V = M_{c2w}^{-1} = \begin{bmatrix} R^T & -R^T eye \\ 0 & 1 \end{bmatrix} = \begin{bmatrix}
r_x & r_y & r_z & -r \cdot eye \\
v_x & v_y & v_z & -v \cdot eye \\
w_x & w_y & w_z & -w \cdot eye \\
0 & 0 & 0 & 1
\end{bmatrix} \\ \quad \\
\text{直观意义: 相机坐标 = 点相对于相机的位置 } (p - eye) \text{ 在 r, v, w 三个方向上的投影 (点积)} \\
x_{cam} = r \cdot (p - eye), \quad y_{cam} = v \cdot (p - eye), \quad z_{cam} = w \cdot (p - eye) \\
\text{检查: eye → 原点; target 在 } -w \text{ 方向, 所以 target → } (0, 0, -|eye - target|)
$$

数学基础: [矩阵的逆](../math_basics/linear_algebra.ipynb#inverse) · [转置](../math_basics/linear_algebra.ipynb#transpose)

In [ ]:
def look_at(eye, target, up):
    r, v, w = camera_basis(eye, target, up)
    R = np.column_stack([r, v, w])
    V = np.eye(4)
    V[:3, :3] = R.T
    V[:3, 3] = -R.T @ eye
    return V

def to_h(p):
    p = np.atleast_2d(p)
    return np.column_stack([p, np.ones(len(p))])

V = look_at(eye, target, up)
M_c2w = np.eye(4)
M_c2w[:3, :3] = R_cam
M_c2w[:3, 3] = eye
print(f'V == inv(M_c2w) ? {np.allclose(V, np.linalg.inv(M_c2w))}')
print(f'eye -> {(V @ to_h(eye).T).T[0, :3]}')
print(f'target -> {(V @ to_h(target).T).T[0, :3]}, -|eye - target| = {-np.linalg.norm(eye - target):.4f}')

rng = np.random.default_rng(1)
world_pts = rng.normal(size=(6, 3))
cam_by_matrix = (V @ to_h(world_pts).T).T[:, :3]
cam_by_dot = np.column_stack([(world_pts - eye) @ r, (world_pts - eye) @ v, (world_pts - eye) @ w])
print(f'矩阵 == 点积投影 ? {np.allclose(cam_by_matrix, cam_by_dot)}')
d_world = np.linalg.norm(world_pts[0] - world_pts[1])
d_cam = np.linalg.norm(cam_by_matrix[0] - cam_by_matrix[1])
print(f'距离保持不变 (刚体变换): 世界 = {d_world:.6f}, 相机 = {d_cam:.6f}')

<a id="mvp"></a>

$$
\text{完整的渲染管线 (MVP Pipeline)} \\
p_{clip} = P \cdot V \cdot M \cdot p_{local} \\
M \text{ (Model): 物体局部坐标 → 世界坐标 (TRS)} \\
V \text{ (View): 世界坐标 → 相机坐标 (lookAt)} \\
P \text{ (Projection): 相机坐标 → 裁剪坐标 (透视投影矩阵, 推导见 perspective\_projection)} \\ \quad \\
\text{透视除法: } p_{ndc} = \frac{(x_c, y_c, z_c)}{w_c} \in [-1, 1]^3 \text{ (} w_c = -z_{cam} \text{, 相机后方的点 } w_c < 0 \text{ 要先裁剪掉)} \\ \quad \\
\text{视口变换 (Viewport Transform): NDC → 屏幕像素, 每个坐标一个一维线性映射} \\
x_{ndc} = -1 \to 0, \ x_{ndc} = 1 \to W \implies x_s = \frac{x_{ndc} + 1}{2} W \\
\text{屏幕的 y 轴向下 (第 0 行在最上面), 所以 } y_{ndc} = 1 \to 0, \ y_{ndc} = -1 \to H \implies y_s = \frac{1 - y_{ndc}}{2} H
$$

数学基础: [矩阵乘法](../math_basics/linear_algebra.ipynb#matrix-multiplication)

In [ ]:
def perspective(fov, aspect, n, f):
    F = 1 / np.tan(np.deg2rad(fov) / 2)
    return np.array([[F / aspect, 0, 0, 0],
                     [0, F, 0, 0],
                     [0, 0, -(f + n) / (f - n), -2 * f * n / (f - n)],
                     [0, 0, -1, 0]])

def model_matrix(t, angle_y, s):
    c, si = np.cos(angle_y), np.sin(angle_y)
    T = np.eye(4); T[:3, 3] = t
    R = np.eye(4); R[:3, :3] = [[c, 0, si], [0, 1, 0], [-si, 0, c]]
    S = np.diag([s, s, s, 1.0])
    return T @ R @ S

def to_screen(local_pts, M, V, P, W, H):
    clip = (P @ V @ M @ to_h(local_pts).T).T
    assert np.all(clip[:, 3] > 0), '有点在相机后面'
    ndc = clip[:, :3] / clip[:, 3:4]
    xs = (ndc[:, 0] + 1) / 2 * W
    ys = (1 - ndc[:, 1]) / 2 * H
    return np.column_stack([xs, ys]), ndc

# 单位立方体的 8 个顶点和 12 条边
cube = np.array([[x, y, z] for x in (-0.5, 0.5) for y in (-0.5, 0.5) for z in (-0.5, 0.5)])
edges = [(i, j) for i in range(8) for j in range(i + 1, 8) if np.sum(np.abs(cube[i] - cube[j])) == 1]
print(f'立方体: {len(cube)} 个顶点, {len(edges)} 条边')

W, H = 640, 480
M = model_matrix(t=[0, 0.5, 0], angle_y=np.deg2rad(20), s=1.5)
P = perspective(50, W / H, 0.1, 100)
screen, ndc = to_screen(cube, M, V, P, W, H)

# 独立验证: 不用 4x4 矩阵, 按 "点积投影到相机坐标 + 相似三角形" 一步步算
world = (M @ to_h(cube).T).T[:, :3]
xc, yc, zc = ((world - eye) @ r), ((world - eye) @ v), ((world - eye) @ w)
F = 1 / np.tan(np.deg2rad(50) / 2)
x_ndc = F / (W / H) * xc / (-zc)
y_ndc = F * yc / (-zc)
print(f'MVP 矩阵 == 逐步手算 ? {np.allclose(ndc[:, :2], np.column_stack([x_ndc, y_ndc]))}')
print(f'所有顶点都在屏幕内 ? {np.all((screen >= 0) & (screen <= [W, H]))}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for ax, cam_eye in zip(axs, [eye, np.array([-3.0, 4.0, 2.0])]):
    s_pts, _ = to_screen(cube, M, look_at(cam_eye, target, up), P, W, H)
    for i, j in edges:
        ax.plot(s_pts[[i, j], 0], s_pts[[i, j], 1], 'b')
    ax.plot([], [], 'b', label=f'eye = {cam_eye}')
    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)      # 屏幕 y 轴向下
    ax.set_aspect('equal')
plot_nicely()
plt.show()

<a id="orbit-camera"></a>

$$
\text{轨道相机 (Orbit Camera): 相机绕目标点做圆周运动, 一直看向目标} \\
eye(\phi) = target + d \, (\cos\phi, \ h, \ \sin\phi) \\
\text{每一帧重新算 lookAt. 物体在屏幕上的投影会变化, 但到相机的距离不变} \\
\text{(物体中心到相机的距离 = 相机坐标的长度, 因为视图矩阵是刚体变换 = 旋转 + 平移, 不改变长度)}
$$

In [ ]:
center_world = (M @ np.array([0, 0, 0, 1.0]))[:3]
fig, axs = plt.subplots(1, 4, figsize=(12, 3))
for ax, phi in zip(axs, np.deg2rad([0, 60, 120, 180])):
    cam = center_world + 5 * np.array([np.cos(phi), 0.5, np.sin(phi)])
    V_o = look_at(cam, center_world, up)
    s_pts, _ = to_screen(cube, M, V_o, P, W, H)
    dist = np.linalg.norm((V_o @ np.append(center_world, 1))[:3])
    print(f'phi = {np.rad2deg(phi):5.1f} deg: 物体中心在相机坐标中的位置 = {np.round((V_o @ np.append(center_world, 1))[:3], 4) + 0.0}, 距离 = {dist:.4f}')
    for i, j in edges:
        ax.plot(s_pts[[i, j], 0], s_pts[[i, j], 1], 'b')
    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)
    ax.set_aspect('equal')
    ax.set_title(f'phi = {np.rad2deg(phi):.0f} deg')
plot_nicely()
plt.show()